In [ ]:
!pip install spreg libpysal

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import spreg
from libpysal.weights import Queen
from sklearn.preprocessing import StandardScaler
import warnings
import os
warnings.filterwarnings('ignore')
from google.colab import drive

# ============================================================
# 1. MONTAR DRIVE Y RUTAS
# ============================================================
drive.mount('/content/drive')
ruta_base = "/content/drive/MyDrive/Analisis_Geoespacial/Proyecto_Curso/Datos/Datos_procesados/"

ruta_zonas = os.path.join(ruta_base, "Zonas_Medellin.gpkg")
ruta_temp = os.path.join(ruta_base, "Temperatura_por_zona.csv")
ruta_arboles = os.path.join(ruta_base, "Arboles_Activos_Medellin_TODOS.gpkg")

# ============================================================
# 2. CARGAR Y PREPARAR DATOS
# ============================================================
zonas = gpd.read_file(ruta_zonas)
df_temp = pd.read_csv(ruta_temp)

# Temperatura promedio por zona
temp_promedio = df_temp.groupby('ID')['Temp_media'].mean().reset_index()

# Unir temperatura a zonas
zonas_temp = zonas.merge(temp_promedio, left_index=True, right_on='ID', how='left')

# Filtrar solo zonas urbanas
zonas_urbano = zonas_temp[zonas_temp['Tipo'] == 'Urbano'].copy()

# Cargar árboles
try:
    arboles = gpd.read_file(ruta_arboles, columns=['geometry'])
except MemoryError:
    arboles = gpd.read_file(ruta_arboles, rows=50000)

# Unión espacial
if arboles.crs != zonas_urbano.crs:
    arboles = arboles.to_crs(zonas_urbano.crs)

arboles_en_zonas = gpd.sjoin(arboles, zonas_urbano, how='inner', predicate='within')

# Contar árboles por zona
conteo_arboles = arboles_en_zonas.groupby('ID').size().reset_index(name='num_arboles')

# Unir conteo a zonas urbanas
zonas_urbano = zonas_urbano.merge(conteo_arboles, on='ID', how='left')
zonas_urbano['num_arboles'] = zonas_urbano['num_arboles'].fillna(0).astype(int)

# Calcular área y densidad
zonas_urbano['area_km2'] = zonas_urbano.geometry.area / 1_000_000
zonas_urbano['densidad_arboles'] = zonas_urbano['num_arboles'] / zonas_urbano['area_km2']

# Eliminar zonas sin temperatura (si hay)
zonas_urbano = zonas_urbano.dropna(subset=['Temp_media', 'densidad_arboles'])

print(f"Zonas urbanas: {len(zonas_urbano)}")
print(f"Densidad media: {zonas_urbano['densidad_arboles'].mean():.2f} arboles/km2")
print(f"Temp media: {zonas_urbano['Temp_media'].mean():.2f} C")

# ============================================================
# 3. DEFINIR VARIABLES
# ============================================================
y = zonas_urbano['Temp_media'].values.reshape((-1, 1))

# Variable predictora
X_vars = ['densidad_arboles']
X = zonas_urbano[X_vars].values

# Estandarizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Matriz de pesos espaciales (Queen)
w = Queen.from_dataframe(zonas_urbano)
w.transform = 'r'

print(f"Matriz de pesos creada: {w.n} observaciones")

# ============================================================
# 4. MODELOS ESPACIALES
# ============================================================

# --- 4.1 MODELO SLX ---
print("\n" + "=" * 60)
print("MODELO SLX (Spatial Lag of X)")
print("=" * 60)

wx = spreg.weights.spatial_lag.lag_spatial(w, X_scaled)
wx = wx.reshape((-1, 1))

slx_exog = np.column_stack([X_scaled, wx])
slx_model = spreg.OLS(y, slx_exog,
                      name_y='Temp_media',
                      name_x=['densidad_arboles', 'w_densidad_arboles'],
                      name_w='Queen')
print(slx_model.summary)

# --- 4.2 MODELO SEM ---
print("\n" + "=" * 60)
print("MODELO SEM (Spatial Error Model)")
print("=" * 60)

sem_model = spreg.GM_Error_Het(y, X_scaled, w=w,
                               name_y='Temp_media',
                               name_x=X_vars,
                               name_w='Queen')
print(sem_model.summary)

# --- 4.3 MODELO SAR-LAG ---
print("\n" + "=" * 60)
print("MODELO SAR-LAG (Spatial Lag Model)")
print("=" * 60)

sar_lag_model = spreg.GM_Lag(y, X_scaled, w=w,
                             name_y='Temp_media',
                             name_x=X_vars,
                             name_w='Queen')
print(sar_lag_model.summary)

# --- 4.4 MODELO SDM (CORREGIDO) ---
print("\n" + "=" * 60)
print("MODELO SDM (Spatial Durbin Model)")
print("=" * 60)

# Para SDM con una sola variable, necesitamos usar X_scaled y wx
# Pero para evitar la singularidad, combinamos en una sola matriz
sdm_exog = np.column_stack([X_scaled, wx])

# Verificar correlación para evitar singularidad
correlacion = np.corrcoef(X_scaled.flatten(), wx.flatten())[0, 1]
print(f"Correlación entre X y wX: {correlacion:.4f}")

if abs(correlacion) > 0.99:
    print("ALTA CORRELACIÓN entre X y wX. SDM puede ser inestable.")
    print("Se recomienda interpretar con cautela o usar solo SAR-Lag o SEM.")

# Intentar SDM
try:
    sdm_model = spreg.GM_Lag(y, sdm_exog, w=w,
                             name_y='Temp_media',
                             name_x=['densidad_arboles', 'w_densidad_arboles'],
                             name_w='Queen')
    print(sdm_model.summary)
except Exception as e:
    print(f"SDM no se pudo ajustar: {e}")
    print("Usando SAR-Lag como alternativa para comparación.")
    sdm_model = sar_lag_model

# ============================================================
# 5. COMPARACIÓN DE MODELOS
# ============================================================
modelos = {
    'SLX': slx_model,
    'SEM': sem_model,
    'SAR-Lag': sar_lag_model,
    'SDM': sdm_model if 'sdm_model' in locals() else sar_lag_model
}

comparacion = []
for nombre, modelo in modelos.items():
    try:
        r2 = modelo.r2 if hasattr(modelo, 'r2') else None
        aic = modelo.aic if hasattr(modelo, 'aic') else None
        log_lik = modelo.llik if hasattr(modelo, 'llik') else None
        comparacion.append({
            'Modelo': nombre,
            'R²': r2,
            'AIC': aic,
            'Log-Lik': log_lik
        })
    except:
        pass

df_comparacion = pd.DataFrame(comparacion)

print("\n" + "=" * 60)
print("COMPARACIÓN DE MODELOS ESPACIALES")
print("=" * 60)
print(df_comparacion.to_string(index=False))

# Recomendación
if not df_comparacion[df_comparacion['AIC'].notna()].empty:
    mejor_modelo = df_comparacion.loc[df_comparacion['AIC'].idxmin()]
    print(f"El modelo con mejor ajuste (menor AIC) es: {mejor_modelo['Modelo']}")
    print(f"   R²: {mejor_modelo['R²']:.4f}")
    print(f"   AIC: {mejor_modelo['AIC']:.2f}")
else:
    print("No se pudo determinar el mejor modelo por AIC.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Zonas urbanas: 271
Densidad media: 1803.78 arboles/km2
Temp media: 36.58 C
Matriz de pesos creada: 271 observaciones

MODELO SLX (Spatial Lag of X)
REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ORDINARY LEAST SQUARES
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :        None
Dependent Variable  :  Temp_media                Number of Observations:         271
Mean dependent var  :     36.5803                Number of Variables   :           3
S.D. dependent var  :      3.3011                Degrees of Freedom    :         268
R-squared           :      0.1284
Adjusted R-squared  :      0.1219
Sum squared residual:     2564.43                F-statistic           :     19.7434
Sigma-square        :       9.569                Prob(F-statistic)     :   1.003e